# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [20]:
!git clone https://github.com/Nikita-Sudarshan/flyrank-ml-starter.git

Cloning into 'flyrank-ml-starter'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 133 (delta 43), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.88 MiB | 10.30 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [21]:
%cd /content/flyrank-ml-starter

/content/flyrank-ml-starter


In [22]:
!ls

 AGENTS.md		 DATA_USE.md	      notebooks		 SETUP.md
 CLAUDE.md		 docs		      outputs		 skills
'Complete Notebook 01'	 flyrank-ml-starter   README.md		 submission
'Complete Notebook 02'	 GUIDE.md	      requirements.txt	 work
 data			 LICENSE	      scripts


In [23]:
import pandas as pd
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


In [24]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [25]:
os.makedirs("work/outputs", exist_ok=True)

print("Output directory ready:", os.path.exists("work/outputs"))

Output directory ready: True


In [26]:
# ### Baseline rule

# Prioritize content for refresh review when it is very stale and still has measurable search visibility.

# A page receives a higher score when:
# - `days_since_last_update >= 181`
# - `impressions_90d >= 500`

# The score uses a simple transparent rule rather than fitted weights. The reason code explains why an item was selected.

# ### Reason code

# - `stale_but_visible` — the content has not been updated for 181+ days and has at least 500 impressions in the trailing 90 days.

# ### Action label

# - `refresh_review` — review the content for a possible refresh.
# - `no_action` — does not meet the baseline conditions.

In [27]:
stale = (df["days_since_last_update"] >= 181).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["score"] = stale * visible * df["impressions_90d"]

df["reason_code"] = "none"
df.loc[(stale == 1) & (visible == 1), "reason_code"] = "stale_but_visible"

df["action"] = "no_action"
df.loc[(stale == 1) & (visible == 1), "action"] = "refresh_review"

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [28]:
# Rank highest-priority items first
ranked = df.sort_values(
    by=["score", "impressions_90d"],
    ascending=[False, False]
).copy()

ranked["rank"] = range(1, len(ranked) + 1)

# Keep the fields needed for the baseline queue
baseline_queue = ranked[
    [
        "rank",
        "content_id",
        "score",
        "action",
        "reason_code",
        "days_since_last_update",
        "impressions_90d"
    ]
].copy()

# Write the required output
output_path = "work/outputs/baseline_action_score.csv"
baseline_queue.to_csv(output_path, index=False)

print(f"Wrote {len(baseline_queue):,} rows to {output_path}")

Wrote 30,000 rows to work/outputs/baseline_action_score.csv


In [29]:
baseline_queue.head(10)

,rank,content_id,score,action,reason_code,days_since_last_update,impressions_90d
16751,1,content_cf56e2e2e282,61678,refresh_review,stale_but_visible,194,61678
16514,2,content_7368877ea310,59472,refresh_review,stale_but_visible,194,59472
7021,3,content_1bfaa38ff26c,25715,refresh_review,stale_but_visible,194,25715
21268,4,content_0a91db491d14,13299,refresh_review,stale_but_visible,193,13299
11489,5,content_5feee3994adb,7812,refresh_review,stale_but_visible,194,7812
12045,6,content_c2d929d83eaa,7558,refresh_review,stale_but_visible,193,7558
698,7,content_b16bd7307b39,4590,refresh_review,stale_but_visible,194,4590
5327,8,content_fe16a55cd13d,4556,refresh_review,stale_but_visible,194,4556
26810,9,content_ecb6215e79fd,4429,refresh_review,stale_but_visible,194,4429
20837,10,content_928af3e22c80,1697,refresh_review,stale_but_visible,193,1697


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [30]:
top20 = ranked.head(20)[
    [
        "rank",
        "content_id",
        "score",
        "action",
        "reason_code",
        "days_since_last_update",
        "impressions_90d",
        "sessions_90d"
    ]
].copy()

top20

,rank,content_id,score,action,reason_code,days_since_last_update,impressions_90d,sessions_90d
16751,1,content_cf56e2e2e282,61678,refresh_review,stale_but_visible,194,61678,119
16514,2,content_7368877ea310,59472,refresh_review,stale_but_visible,194,59472,82
7021,3,content_1bfaa38ff26c,25715,refresh_review,stale_but_visible,194,25715,80
21268,4,content_0a91db491d14,13299,refresh_review,stale_but_visible,193,13299,78
11489,5,content_5feee3994adb,7812,refresh_review,stale_but_visible,194,7812,5
12045,6,content_c2d929d83eaa,7558,refresh_review,stale_but_visible,193,7558,25
698,7,content_b16bd7307b39,4590,refresh_review,stale_but_visible,194,4590,4
5327,8,content_fe16a55cd13d,4556,refresh_review,stale_but_visible,194,4556,42
26810,9,content_ecb6215e79fd,4429,refresh_review,stale_but_visible,194,4429,12
20837,10,content_928af3e22c80,1697,refresh_review,stale_but_visible,193,1697,3


In [31]:
top20_review = top20.copy()

top20_review["confidence_note"] = ""
top20_review["what_would_make_it_wrong"] = ""

top20_review.loc[
    top20_review["action"] == "refresh_review",
    "confidence_note"
] = "Higher priority because the item is very stale and still has measurable visibility."

top20_review.loc[
    top20_review["action"] == "refresh_review",
    "what_would_make_it_wrong"
] = "The traffic may be temporary, low-quality, or unrelated to content freshness; updating may not improve performance."

top20_review.loc[
    top20_review["action"] == "no_action",
    "confidence_note"
] = "Low confidence as a no-action decision because the rule gives zero score."

top20_review.loc[
    top20_review["action"] == "no_action",
    "what_would_make_it_wrong"
] = "The item may still deserve attention for reasons not captured by this simple staleness-and-visibility rule."

top20_review[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
16751,1,content_cf56e2e2e282,refresh_review,stale_but_visible,Higher priority because the item is very stale...,"The traffic may be temporary, low-quality, or ..."
16514,2,content_7368877ea310,refresh_review,stale_but_visible,Higher priority because the item is very stale...,"The traffic may be temporary, low-quality, or ..."
7021,3,content_1bfaa38ff26c,refresh_review,stale_but_visible,Higher priority because the item is very stale...,"The traffic may be temporary, low-quality, or ..."
21268,4,content_0a91db491d14,refresh_review,stale_but_visible,Higher priority because the item is very stale...,"The traffic may be temporary, low-quality, or ..."
11489,5,content_5feee3994adb,refresh_review,stale_but_visible,Higher priority because the item is very stale...,"The traffic may be temporary, low-quality, or ..."
12045,6,content_c2d929d83eaa,refresh_review,stale_but_visible,Higher priority because the item is very stale...,"The traffic may be temporary, low-quality, or ..."
698,7,content_b16bd7307b39,refresh_review,stale_but_visible,Higher priority because the item is very stale...,"The traffic may be temporary, low-quality, or ..."
5327,8,content_fe16a55cd13d,refresh_review,stale_but_visible,Higher priority because the item is very stale...,"The traffic may be temporary, low-quality, or ..."
26810,9,content_ecb6215e79fd,refresh_review,stale_but_visible,Higher priority because the item is very stale...,"The traffic may be temporary, low-quality, or ..."
20837,10,content_928af3e22c80,refresh_review,stale_but_visible,Higher priority because the item is very stale...,"The traffic may be temporary, low-quality, or ..."


In [32]:
# ### Top-20 review

# The baseline prioritizes stale content that still has at least 500 impressions in the trailing 90 days.

# Most of the top-ranked items have substantially more than the minimum visibility threshold, so the rule is selecting pages with meaningful observed visibility. However, the rule is not a guarantee that a refresh will improve performance.

# Rank 17 is a weak pick: it has only 533 impressions and 2 sessions despite meeting the 500-impression threshold. This shows that the threshold is somewhat blunt and can include items with very low observed activity.

# For every row, the action is `refresh_review` when the rule is met and `no_action` otherwise. The main failure mode is that traffic may not be related to content freshness, so a refresh may not improve performance.

In [33]:
ranked.loc[
    ranked["rank"] == 17,
    [
        "rank",
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "sessions_90d",
        "score",
        "action",
        "reason_code"
    ]
]

,rank,content_id,days_since_last_update,impressions_90d,sessions_90d,score,action,reason_code
3507,17,content_074ba6ead17b,183,533,2,533,refresh_review,stale_but_visible


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [34]:
# ### Weak pick

# Rank 17 is a weak pick for this baseline. It has 183 days since its last update and 533 impressions, so it barely passes the 500-impression visibility threshold. However, it has only 2 sessions in the trailing 90 days. This suggests the simple threshold can prioritize an item with very limited observed activity.

# ### Leakage check

# The baseline uses only:
# - `days_since_last_update`
# - `impressions_90d`

# It does not use `trend_pct`, `trend_direction`, or any label-derived field. It also does not use client or content IDs as predictive features.

# The rule uses trailing 90-day observations and does not use future-window information. No product flags are used as inputs to the score.

In [35]:
rule_features = [
    "days_since_last_update",
    "impressions_90d"
]

print("Features used by baseline:")
print(rule_features)

for col in rule_features:
    print(f"{col}: {df[col].isna().sum()} missing values")

for forbidden in ["trend_pct", "trend_direction", "is_declining_label"]:
    print(f"{forbidden} used in score: {forbidden in rule_features}")

Features used by baseline:
['days_since_last_update', 'impressions_90d']
days_since_last_update: 0 missing values
impressions_90d: 0 missing values
trend_pct used in score: False
trend_direction used in score: False
is_declining_label used in score: False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.